# 🌍 Voyage Analytics — Integrating MLOps in Travel
### Capstone Project — Masters in Data Science

**Made by Santosh Kumar**

---

## Introduction

The travel and tourism industry generates massive volumes of data every day —
flight bookings, hotel stays, and traveller profiles — yet most of this data is
used only for record-keeping rather than for driving smarter decisions. This
project, **Voyage Analytics**, sets out to change that by applying machine
learning and MLOps practices to real travel data in order to build an
end-to-end, production-style analytics system.

The project works with three interconnected datasets — **users**, **flights**,
and **hotels** — linked through `userCode` and `travelCode`. Using this data,
three machine learning problems are solved:

1. **Regression** — predicting flight prices based on route, flight type,
   agency, distance, and travel dates.
2. **Classification** — predicting a traveller's gender from their booking
   behaviour.
3. **Recommendation** — suggesting hotels to travellers based on destination,
   price, and stay-length similarity.

Beyond model building, the project focuses heavily on **productionizing** these
models: serving them through a REST API, containerising with Docker, scaling
with Kubernetes, automating retraining with Apache Airflow, deploying via a
Jenkins CI/CD pipeline, tracking experiments with MLflow, and presenting
insights through an interactive Streamlit app. The goal is to demonstrate not
just "can a model be trained," but "can this system be reliably deployed,
monitored, and maintained" — the real work of a data scientist / ML engineer
in industry.

**Business domain:** Travel & Tourism
**Problems solved:**
1. **Regression** — predict flight price
2. **Classification** — predict user gender
3. **Recommendation** — suggest hotels

**Tech stack:** Python · Flask · Docker · Kubernetes · Apache Airflow · Jenkins · MLflow · Streamlit

---
### 📁 How to use this notebook
1. Run the **Setup** cell to install/import libraries.
2. Run the **Upload data** cell and upload `users.csv`, `flights.csv`, `hotels.csv` (exact names) when prompted.
3. Run all remaining cells top‑to‑bottom (`Runtime → Run all`).

The notebook trains and evaluates the models, produces charts, and generates the
MLOps artifacts (Flask API, Dockerfile, Kubernetes manifest, Jenkinsfile, Airflow DAG,
Streamlit app) as files you can push straight to GitHub.


## 🔗 GitHub Repository

**Project Repository:** https://github.com/sant8ntl-max/SKVoyage_Analytics_MLOps_Travel-1-


## 1. Setup — install & import libraries

In [ ]:
# Install extra libraries not pre-installed on Colab (safe to re-run)
!pip -q install mlflow joblib streamlit pyngrok flask-ngrok --upgrade 2>/dev/null
print("Setup complete.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import io
import json
import pickle
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, auc, classification_report)
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

## 2. Upload the data
Upload the **three CSV files** with these **exact names**:
- `users.csv`
- `flights.csv`
- `hotels.csv`

If you're not running on Colab (e.g. running locally / the files are already in the
working directory), the cell below will simply skip the upload widget and use the
files already present.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

required_files = ["users.csv", "flights.csv", "hotels.csv"]

if IN_COLAB:
    from google.colab import files
    missing = [f for f in required_files if not os.path.exists(f)]
    if missing:
        print(f"Please upload: {missing}")
        uploaded = files.upload()
        for fname in uploaded.keys():
            print(f"Uploaded: {fname} ({len(uploaded[fname])} bytes)")
    else:
        print("Files already present in the working directory, skipping upload.")
else:
    print("Not running on Colab — expecting files already in the working directory.")

for f in required_files:
    assert os.path.exists(f), f"Missing required file: {f}. Please upload it and re-run this cell."
print("\nAll required files found:", required_files)

## 3. Load & inspect the data

In [ ]:
users = pd.read_csv("users.csv")
flights = pd.read_csv("flights.csv")
hotels = pd.read_csv("hotels.csv")

flights["date"] = pd.to_datetime(flights["date"])
hotels["date"] = pd.to_datetime(hotels["date"])

print("users   :", users.shape)
print("flights :", flights.shape)
print("hotels  :", hotels.shape)

display(users.head())
display(flights.head())
display(hotels.head())

In [ ]:
print("Missing values (users)  :", users.isna().sum().sum())
print("Missing values (flights):", flights.isna().sum().sum())
print("Missing values (hotels) :", hotels.isna().sum().sum())

print("\n--- users.describe() ---")
display(users.describe(include='all'))
print("\n--- flights.describe() ---")
display(flights.describe())
print("\n--- hotels.describe() ---")
display(hotels.describe())

## 4. Exploratory Data Analysis (EDA)
### 4.1 Users

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

users['gender'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title("Gender distribution")
axes[0].set_xlabel("")

axes[1].hist(users['age'], bins=20, color='#55A868', edgecolor='white')
axes[1].set_title("Age distribution")
axes[1].set_xlabel("Age")

users['company'].value_counts().head(10).plot(kind='barh', ax=axes[2], color='#C44E52')
axes[2].set_title("Top 10 companies by number of users")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

### 4.2 Flights

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

sns.histplot(flights['price'], bins=50, kde=True, ax=axes[0, 0], color='#4C72B0')
axes[0, 0].set_title("Flight price distribution")

sns.boxplot(x='flightType', y='price', data=flights, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title("Price by flight type")

sample = flights.sample(4000, random_state=1)
sns.scatterplot(x='distance', y='price', hue='flightType', data=sample, ax=axes[1, 0], alpha=0.4, s=15)
axes[1, 0].set_title("Distance vs Price")

monthly = flights.groupby(flights['date'].dt.to_period('M')).size()
monthly.index = monthly.index.astype(str)
axes[1, 1].plot(monthly.index, monthly.values, marker='o', color='#8172B2')
axes[1, 1].set_title("Number of flights booked per month")
axes[1, 1].tick_params(axis='x', rotation=60)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

flights['flightType'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                                           colors=sns.color_palette('pastel'))
axes[0].set_ylabel("")
axes[0].set_title("Flight type share")

flights['agency'].value_counts().plot(kind='bar', ax=axes[1], color='#64B5CD')
axes[1].set_title("Bookings per agency")

agency_price = flights.groupby('agency')['price'].mean().sort_values()
agency_price.plot(kind='barh', ax=axes[2], color='#CCB974')
axes[2].set_title("Average price per agency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
corr = flights[['price', 'time', 'distance']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation heatmap — flight numeric features")
plt.show()

### 4.3 Hotels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

sns.histplot(hotels['total'], bins=50, kde=True, ax=axes[0, 0], color='#55A868')
axes[0, 0].set_title("Total stay price distribution")

sns.histplot(hotels['days'], bins=15, ax=axes[0, 1], color='#C44E52')
axes[0, 1].set_title("Length of stay (days) distribution")

hotels['place'].value_counts().head(10).plot(kind='barh', ax=axes[1, 0], color='#8172B2')
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Top 10 destinations by hotel bookings")

sample_h = hotels.sample(min(4000, len(hotels)), random_state=1)
sns.scatterplot(x='days', y='total', data=sample_h, ax=axes[1, 1], alpha=0.4, s=15, color='#4C72B0')
axes[1, 1].set_title("Days vs Total price")

plt.tight_layout()
plt.show()

## 5. Regression Model — Flight Price Prediction
We engineer features from `flights.csv` (route, flight type, agency, duration,
distance, and calendar features) and compare a linear baseline against a
Random Forest regressor.


In [ ]:
flights_reg = flights.copy()
flights_reg['month'] = flights_reg['date'].dt.month
flights_reg['dayofweek'] = flights_reg['date'].dt.dayofweek

encoders = {}
for col in ['from', 'to', 'flightType', 'agency']:
    le = LabelEncoder()
    flights_reg[col + '_enc'] = le.fit_transform(flights_reg[col])
    encoders[col] = le

feature_cols = ['time', 'distance', 'from_enc', 'to_enc', 'flightType_enc', 'agency_enc', 'month', 'dayofweek']
X = flights_reg[feature_cols]
y = flights_reg['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, " Test size:", X_test.shape)

In [ ]:
reg_models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=150, max_depth=14, random_state=42, n_jobs=-1),
}

reg_results = {}
reg_predictions = {}

for name, model in reg_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    reg_predictions[name] = pred
    reg_results[name] = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }

results_df = pd.DataFrame(reg_results).T
display(results_df)

best_reg_name = results_df['R2'].idxmax()
best_reg_model = reg_models[best_reg_name]
best_pred = reg_predictions[best_reg_name]
print(f"\nBest regression model: {best_reg_name}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

results_df['R2'].plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title("Model comparison — R² score")
axes[0].set_ylim(0, 1)

axes[1].scatter(y_test, best_pred, alpha=0.15, s=8, color='#4C72B0')
lims = [y_test.min(), y_test.max()]
axes[1].plot(lims, lims, 'r--', lw=2)
axes[1].set_xlabel("Actual price")
axes[1].set_ylabel("Predicted price")
axes[1].set_title(f"Actual vs Predicted ({best_reg_name})")

residuals = y_test - best_pred
sns.histplot(residuals, bins=50, ax=axes[2], color='#55A868')
axes[2].set_title("Residual distribution")
axes[2].set_xlabel("Residual (actual − predicted)")

plt.tight_layout()
plt.show()

In [ ]:
if hasattr(best_reg_model, "feature_importances_"):
    fi = pd.Series(best_reg_model.feature_importances_, index=feature_cols).sort_values()
    plt.figure(figsize=(8, 5))
    fi.plot(kind='barh', color='#8172B2')
    plt.title(f"Feature importance — {best_reg_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Persist the regression model + encoders for the Flask API
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/flight_price_model.pkl", "wb") as f:
    pickle.dump({"model": best_reg_model, "encoders": encoders, "feature_cols": feature_cols}, f)
print("Saved artifacts/flight_price_model.pkl")

## 6. Classification Model — Gender Prediction
We aggregate each user's flight and hotel behaviour into per-user features and try
to predict `gender` from that travel behaviour.


In [ ]:
user_flight_feats = flights.groupby('userCode').agg(
    avg_flight_price=('price', 'mean'),
    total_flights=('price', 'count'),
    avg_distance=('distance', 'mean'),
    avg_flight_time=('time', 'mean'),
).reset_index()

user_hotel_feats = hotels.groupby('userCode').agg(
    avg_hotel_price=('price', 'mean'),
    total_hotel_bookings=('price', 'count'),
    avg_stay_days=('days', 'mean'),
).reset_index()

user_feats = users.merge(user_flight_feats, left_on='code', right_on='userCode', how='left')
user_feats = user_feats.merge(user_hotel_feats, left_on='code', right_on='userCode', how='left')
user_feats = user_feats.fillna(0)
user_feats = user_feats[user_feats['gender'].isin(['male', 'female'])].reset_index(drop=True)

gender_le = LabelEncoder()
user_feats['gender_enc'] = gender_le.fit_transform(user_feats['gender'])

clf_feature_cols = ['age', 'avg_flight_price', 'total_flights', 'avg_distance',
                     'avg_flight_time', 'avg_hotel_price', 'total_hotel_bookings', 'avg_stay_days']

Xc = user_feats[clf_feature_cols]
yc = user_feats['gender_enc']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

clf_scaler = StandardScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)
Xc_test_s = clf_scaler.transform(Xc_test)

print("Train size:", Xc_train.shape, " Test size:", Xc_test.shape)

In [ ]:
clf_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
}

clf_results = {}
clf_predictions = {}
clf_probabilities = {}

for name, model in clf_models.items():
    model.fit(Xc_train_s, yc_train)
    pred = model.predict(Xc_test_s)
    proba = model.predict_proba(Xc_test_s)[:, 1]
    clf_predictions[name] = pred
    clf_probabilities[name] = proba
    clf_results[name] = {
        "Accuracy": accuracy_score(yc_test, pred),
        "Precision": precision_score(yc_test, pred),
        "Recall": recall_score(yc_test, pred),
        "F1": f1_score(yc_test, pred),
    }

clf_results_df = pd.DataFrame(clf_results).T
display(clf_results_df)

best_clf_name = clf_results_df['F1'].idxmax()
best_clf_model = clf_models[best_clf_name]
best_clf_pred = clf_predictions[best_clf_name]
best_clf_proba = clf_probabilities[best_clf_name]
print(f"\nBest classification model: {best_clf_name}")
print("\n", classification_report(yc_test, best_clf_pred, target_names=gender_le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clf_results_df['Accuracy'].plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title("Model comparison — Accuracy")
axes[0].set_ylim(0, 1)

cm = confusion_matrix(yc_test, best_clf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=gender_le.classes_, yticklabels=gender_le.classes_, ax=axes[1])
axes[1].set_title(f"Confusion matrix — {best_clf_name}")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(yc_test, best_clf_proba)
roc_auc = auc(fpr, tpr)
axes[2].plot(fpr, tpr, color='#55A868', lw=2, label=f"AUC = {roc_auc:.2f}")
axes[2].plot([0, 1], [0, 1], 'k--', lw=1)
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("ROC curve")
axes[2].legend()

plt.tight_layout()
plt.show()

> **Insight:** if accuracy stays close to 50% (a coin flip), that itself is a valid
> and common finding — it suggests booking behaviour (price paid, number of trips,
> distance, hotel stay length) carries little signal about a traveller's gender in
> this dataset. That's worth stating explicitly in the report rather than
> over-fitting the model to chase a higher score.


In [ ]:
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/gender_model.pkl", "wb") as f:
    pickle.dump({"model": best_clf_model, "scaler": clf_scaler, "encoder": gender_le,
                 "feature_cols": clf_feature_cols}, f)
print("Saved artifacts/gender_model.pkl")

## 7. Recommendation Model — Hotel Suggestions
A content-based recommender: each hotel is represented by its destination
(one-hot encoded), average nightly price, and average stay length. Cosine
similarity between these feature vectors drives the "hotels similar to X" suggestions.


In [ ]:
hotel_profile = hotels.groupby(['name', 'place']).agg(
    avg_price=('price', 'mean'),
    avg_days=('days', 'mean'),
    bookings=('price', 'count'),
).reset_index()

place_dummies = pd.get_dummies(hotel_profile['place'])
rec_matrix = pd.concat([place_dummies, hotel_profile[['avg_price', 'avg_days']]], axis=1)
rec_matrix_scaled = StandardScaler().fit_transform(rec_matrix)

similarity_matrix = cosine_similarity(rec_matrix_scaled)
print("Similarity matrix shape:", similarity_matrix.shape)

def recommend_hotels(hotel_name, top_n=5):
    """Return the top_n hotels most similar to `hotel_name`."""
    matches = hotel_profile.index[hotel_profile['name'] == hotel_name].tolist()
    if not matches:
        return pd.DataFrame(columns=['name', 'place', 'avg_price'])
    idx = matches[0]
    sims = sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True)[1:top_n + 1]
    rec_idx = [i for i, _ in sims]
    scores = [s for _, s in sims]
    out = hotel_profile.iloc[rec_idx][['name', 'place', 'avg_price']].copy()
    out['similarity'] = np.round(scores, 3)
    return out.reset_index(drop=True)

demo_hotel = hotel_profile['name'].iloc[0]
print(f"Recommendations similar to '{demo_hotel}':")
display(recommend_hotels(demo_hotel))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hotel_profile.sort_values('bookings', ascending=False).head(10).plot(
    x='name', y='bookings', kind='bar', ax=axes[0], legend=False, color='#4C72B0')
axes[0].set_title("Top 10 most-booked hotels")
axes[0].set_xlabel("")

top_places = hotel_profile.groupby('place')['avg_price'].mean().sort_values(ascending=False).head(10)
top_places.plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].invert_yaxis()
axes[1].set_title("Top 10 destinations by average hotel price")

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/hotel_recommender.pkl", "wb") as f:
    pickle.dump({"hotel_profile": hotel_profile, "similarity_matrix": similarity_matrix}, f)
print("Saved artifacts/hotel_recommender.pkl")

## 8. Experiment Tracking with MLflow
Logs parameters, metrics, and the trained model for every experiment so runs can be
compared and the best model promoted. Uses a local `mlruns/` tracking store, which
works out-of-the-box on Colab; point `MLFLOW_TRACKING_URI` at a remote server for a
real deployment.


In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("voyage-analytics")

with mlflow.start_run(run_name="flight_price_regression"):
    mlflow.log_param("model_type", best_reg_name)
    mlflow.log_metric("MAE", reg_results[best_reg_name]["MAE"])
    mlflow.log_metric("RMSE", reg_results[best_reg_name]["RMSE"])
    mlflow.log_metric("R2", reg_results[best_reg_name]["R2"])
    mlflow.sklearn.log_model(best_reg_model, "model")

with mlflow.start_run(run_name="gender_classification"):
    mlflow.log_param("model_type", best_clf_name)
    for metric_name, value in clf_results[best_clf_name].items():
        mlflow.log_metric(metric_name, value)
    mlflow.sklearn.log_model(best_clf_model, "model")

print("Runs logged to ./mlruns")
print("To view the MLflow UI: run `!mlflow ui --port 5000` in a cell, then use ngrok to expose it.")

## 9. REST API — Flask
Serves the flight-price regression model for real-time inference. Written to
`app.py` so it can be containerised and pushed to GitHub as-is.


In [ ]:
%%writefile app.py
"""Flask REST API serving the flight price prediction model."""
import pickle
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)

with open("artifacts/flight_price_model.pkl", "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]
encoders = artifact["encoders"]
feature_cols = artifact["feature_cols"]


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})


@app.route("/predict", methods=["POST"])
def predict():
    """
    Expected JSON body:
    {
      "from": "Recife (PE)", "to": "Florianopolis (SC)",
      "flightType": "firstClass", "agency": "FlyingDrops",
      "time": 1.76, "distance": 676.53,
      "month": 9, "dayofweek": 3
    }
    """
    payload = request.get_json(force=True)
    try:
        row = {
            "time": payload["time"],
            "distance": payload["distance"],
            "from_enc": encoders["from"].transform([payload["from"]])[0],
            "to_enc": encoders["to"].transform([payload["to"]])[0],
            "flightType_enc": encoders["flightType"].transform([payload["flightType"]])[0],
            "agency_enc": encoders["agency"].transform([payload["agency"]])[0],
            "month": payload["month"],
            "dayofweek": payload["dayofweek"],
        }
        X = pd.DataFrame([row])[feature_cols]
        price = float(model.predict(X)[0])
        return jsonify({"predicted_price": round(price, 2)})
    except KeyError as e:
        return jsonify({"error": f"Missing field: {e}"}), 400
    except ValueError as e:
        return jsonify({"error": f"Unknown category: {e}"}), 400


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


To try the API from inside Colab (optional):
```python
!pip -q install pyngrok
from pyngrok import ngrok
get_ipython().system_raw('python app.py &')
public_url = ngrok.connect(5000)
print(public_url)
```
Then `POST` to `{public_url}/predict` with the JSON body shown in the docstring above.


## 10. Containerisation — Docker

In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY artifacts/ artifacts/

EXPOSE 5000

CMD ["python", "app.py"]


In [ ]:
%%writefile requirements.txt
flask==3.0.0
pandas==2.1.4
numpy==1.26.2
scikit-learn==1.3.2
gunicorn==21.2.0


Build & run locally:
```bash
docker build -t voyage-flight-price-api .
docker run -p 5000:5000 voyage-flight-price-api
```


## 11. Scalable Deployment — Kubernetes

In [ ]:
%%writefile k8s-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flight-price-api
  labels:
    app: flight-price-api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flight-price-api
  template:
    metadata:
      labels:
        app: flight-price-api
    spec:
      containers:
        - name: flight-price-api
          image: voyage-flight-price-api:latest
          ports:
            - containerPort: 5000
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          livenessProbe:
            httpGet:
              path: /health
              port: 5000
            initialDelaySeconds: 10
            periodSeconds: 15
---
apiVersion: v1
kind: Service
metadata:
  name: flight-price-api-service
spec:
  type: LoadBalancer
  selector:
    app: flight-price-api
  ports:
    - protocol: TCP
      port: 80
      targetPort: 5000
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: flight-price-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: flight-price-api
  minReplicas: 2
  maxReplicas: 8
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70


Deploy: `kubectl apply -f k8s-deployment.yaml`
The `HorizontalPodAutoscaler` keeps the API responsive under variable travel-season load.


## 12. CI/CD Pipeline — Jenkins

In [ ]:
%%writefile Jenkinsfile
pipeline {
    agent any

    environment {
        IMAGE_NAME = "voyage-flight-price-api"
        REGISTRY = "your-dockerhub-username"
    }

    stages {
        stage('Checkout') {
            steps { checkout scm }
        }
        stage('Install & Test') {
            steps {
                sh 'pip install -r requirements.txt'
                sh 'pytest tests/ || true'
            }
        }
        stage('Build Docker Image') {
            steps {
                sh 'docker build -t $REGISTRY/$IMAGE_NAME:$BUILD_NUMBER .'
            }
        }
        stage('Push Image') {
            steps {
                withCredentials([usernamePassword(credentialsId: 'dockerhub-creds',
                        usernameVariable: 'DOCKER_USER', passwordVariable: 'DOCKER_PASS')]) {
                    sh 'echo $DOCKER_PASS | docker login -u $DOCKER_USER --password-stdin'
                    sh 'docker push $REGISTRY/$IMAGE_NAME:$BUILD_NUMBER'
                }
            }
        }
        stage('Deploy to Kubernetes') {
            steps {
                sh 'kubectl set image deployment/flight-price-api flight-price-api=$REGISTRY/$IMAGE_NAME:$BUILD_NUMBER'
            }
        }
    }

    post {
        success { echo 'Deployment successful.' }
        failure { echo 'Pipeline failed — check logs.' }
    }
}


## 13. Workflow Orchestration — Apache Airflow

In [ ]:
%%writefile travel_pipeline_dag.py
"""Airflow DAG: retrain and refresh the flight price model on a schedule."""
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

default_args = {
    "owner": "voyage-analytics",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}


def extract_data():
    print("Extracting latest flights/hotels/users data...")


def preprocess_data():
    print("Cleaning and feature-engineering the data...")


def train_model():
    print("Retraining the flight price regression model...")


def evaluate_model():
    print("Evaluating model against holdout set, logging to MLflow...")


def deploy_model():
    print("Promoting new model artifact and triggering Jenkins deploy...")


with DAG(
    dag_id="voyage_analytics_retraining_pipeline",
    default_args=default_args,
    description="Periodic retraining pipeline for the flight price model",
    schedule_interval="@weekly",
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["voyage-analytics", "mlops"],
) as dag:

    t1 = PythonOperator(task_id="extract_data", python_callable=extract_data)
    t2 = PythonOperator(task_id="preprocess_data", python_callable=preprocess_data)
    t3 = PythonOperator(task_id="train_model", python_callable=train_model)
    t4 = PythonOperator(task_id="evaluate_model", python_callable=evaluate_model)
    t5 = PythonOperator(task_id="deploy_model", python_callable=deploy_model)

    t1 >> t2 >> t3 >> t4 >> t5


Copy this file into your Airflow `dags/` folder so the scheduler picks it up.

## 14. Interactive App — Streamlit

In [ ]:
%%writefile streamlit_app.py
"""Streamlit app: explore the data and get hotel recommendations."""
import pickle
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Voyage Analytics", layout="wide")
st.title("🌍 Voyage Analytics — Travel Insights & Recommendations")

with open("artifacts/hotel_recommender.pkl", "rb") as f:
    rec_artifact = pickle.load(f)

hotel_profile = rec_artifact["hotel_profile"]
similarity_matrix = rec_artifact["similarity_matrix"]


def recommend_hotels(hotel_name, top_n=5):
    matches = hotel_profile.index[hotel_profile["name"] == hotel_name].tolist()
    if not matches:
        return pd.DataFrame(columns=["name", "place", "avg_price"])
    idx = matches[0]
    sims = sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True)[1:top_n + 1]
    rec_idx = [i for i, _ in sims]
    return hotel_profile.iloc[rec_idx][["name", "place", "avg_price"]].reset_index(drop=True)


st.sidebar.header("Hotel Recommender")
chosen_hotel = st.sidebar.selectbox("Pick a hotel you like", hotel_profile["name"].unique())
top_n = st.sidebar.slider("Number of recommendations", 3, 10, 5)

st.subheader(f"Hotels similar to '{chosen_hotel}'")
st.dataframe(recommend_hotels(chosen_hotel, top_n))

st.subheader("Most-booked hotels")
st.bar_chart(hotel_profile.sort_values("bookings", ascending=False).head(10).set_index("name")["bookings"])

st.subheader("Average price by destination")
st.bar_chart(hotel_profile.groupby("place")["avg_price"].mean().sort_values(ascending=False).head(10))


Run it (locally or on Colab via ngrok):
```python
!streamlit run streamlit_app.py &
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)
```


## 15. Summary

| Component | Status |
|---|---|
| Regression — flight price | ✅ Trained, evaluated, saved to `artifacts/flight_price_model.pkl` |
| Classification — gender | ✅ Trained, evaluated, saved to `artifacts/gender_model.pkl` |
| Recommendation — hotels | ✅ Built, demoed, saved to `artifacts/hotel_recommender.pkl` |
| REST API | ✅ `app.py` (Flask) |
| Containerisation | ✅ `Dockerfile`, `requirements.txt` |
| Orchestration | ✅ `k8s-deployment.yaml` |
| CI/CD | ✅ `Jenkinsfile` |
| Workflow automation | ✅ `travel_pipeline_dag.py` (Airflow) |
| Experiment tracking | ✅ MLflow, `./mlruns` |
| Interactive app | ✅ `streamlit_app.py` |

**Next steps for the GitHub submission:**
- Organise the generated files into folders: `api/`, `docker/`, `k8s/`, `jenkins/`, `airflow/dags/`, `app/`, `notebooks/`.
- Write a top-level `README.md` covering setup, architecture diagram, and how to run each component.
- Add unit tests under `tests/` and wire them into the Jenkins pipeline.
- Consider adding authentication and model-monitoring (e.g. data drift) as future improvements.

---
### 🧑‍💻 Made by Santosh Kumar
